<a href="https://colab.research.google.com/github/force23airr/Option_trading/blob/main/6_multiplereg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy
import pandas
import scipy.stats
# import statsmodels modules that enable us to implement OLS
import statsmodels.formula.api as smf
import statsmodels.api as sm
from google.colab import files

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)
# if there is any error in this step (pandas.read_excel)
# try opening the Excel file on your local computer and saving it again.

# importing the 'Growth.xlsx' data
dataset.head()

(You can download the '[Growth.xlsx](https://docs.google.com/spreadsheets/d/14tibZidGVVrmNnfR3_FyLJgjye4B-gRN/edit?usp=drive_link&ouid=110351823593825164156&rtpof=true&sd=true)' and '[birthweight_smoking.xlsx](https://docs.google.com/spreadsheets/d/1PSQzHNwPfovfmVIYHlzY8aBVFW-je9Pc/edit?usp=drive_link&ouid=110351823593825164156&rtpof=true&sd=true)' to import.)

#### **Estimation**

We have already known how to run a simple regression:

In [ ]:
mod = smf.ols(formula='growth ~ tradeshare', data=dataset)
result = mod.fit()
print(result.summary())

Suppose our multiple regression model is $$growth_{i}=\beta_{0}+\beta_{1}tradeshare_{i}+\beta_{2} yearsschool_{i} + u_{i} \, .$$

To do the OLS estimation of this model based on the imported data, you just need to:

In [ ]:
mod = smf.ols(formula='growth ~ tradeshare + yearsschool', data=dataset)
result = mod.fit()
print(result.summary())

So, the estimated model is
$$\widehat{growth}_{i}=-0.37+ 2.33 \, tradeshare_{i}+0.25 yearsschool_{i} \, .$$

And, don't forget that, if these are the final results you will report, in normal circumstances, you want to have the estimated standard errors adjusted for heteroskedasticity:

In [ ]:
mod = smf.ols(formula='growth ~ tradeshare + yearsschool', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

In [ ]:
(2.3313-3)/0.596

In [ ]:
print(result2.t_test('tradeshare=3'))

In [ ]:
2.3313+scipy.stats.t.ppf(0.975, 65-3)*0.596

#### **The dummy variable trap**

Let's look at another data set to investigate the issue of the dummy variable trap.

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)
# if there is any error in this step (pandas.read_excel)
# try opening the Excel file on your local computer and saving it again.

# importing the 'birthweight_smoking.xlsx' data
dataset.head()

In [ ]:
mod = smf.ols(formula='birthweight ~ smoker+drinks+unmarried+educ+age', data=dataset)
result = mod.fit()
print(result.summary())

If you read the [data description](https://drive.google.com/file/d/1SYfbF_FiY0l84_2I9nOrkuIaCWaQDWpb/view?usp=drive_link), you will know that there are four dummy variables named `tripre1`, `tripre2`, `tripre3`, `tripre0`. `tripre1` equals to $1$ if a prospective mother's 1st prenatal care visit occurred in the 1st trimester of her pregnancy; `tripre2` equals to $1$ if it occurred in the 2nd trimester of her pregnancy; `tripre3` equals to $1$ if in the 3rd. `tripre0` equals to $1$ if the mother paid no prenatal visits at all.

In [ ]:
mod = smf.ols(formula='birthweight ~ tripre0+tripre1+tripre2+tripre3', data=dataset)
result = mod.fit()
print(result.summary())

In [ ]:
check = dataset[['tripre0','tripre1','tripre2','tripre3']].sum(axis=1) == 1
print(check)
print(check.unique())

In [ ]:
mod = smf.ols(formula='birthweight ~ tripre1+tripre2+tripre3', data=dataset)
result = mod.fit()
print(result.summary())

In [ ]:
mod = smf.ols(formula='birthweight ~ 0+tripre0+tripre1+tripre2+tripre3', data=dataset)
result = mod.fit()
print(result.summary())

In [ ]:
mod = smf.ols(formula='birthweight ~ smoker+drinks+unmarried+educ+age+tripre0+tripre1+tripre2+tripre3', data=dataset)
result = mod.fit()
print(result.summary())

In [ ]:
mod = smf.ols(formula='birthweight ~ smoker+drinks+unmarried+educ+age+tripre1+tripre2+tripre3', data=dataset)
result = mod.fit()
print(result.summary())

In [ ]:
mod = smf.ols(formula='birthweight ~ 0+smoker+drinks+unmarried+educ+age+tripre0+tripre1+tripre2+tripre3', data=dataset)
result = mod.fit()
print(result.summary())

#### **Joint hypotheses tests**

(You can download the '[californiatestscores.xlsx](https://docs.google.com/spreadsheets/d/1UXKWvxDId-whK0nWJsWzRJZfrr0WltoY/edit?usp=drive_link&ouid=110351823593825164156&rtpof=true&sd=true)' to import.)

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)

# importing the 'californiatestscores.xlsx' data
dataset.head()

In [ ]:
mod = smf.ols(formula='testscr ~ str + el_pct', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

In [ ]:
dataset['expn_stu'] = dataset['expn_stu'] / 1000

In [ ]:
mod = smf.ols(formula='testscr ~ str + expn_stu + el_pct', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

The above regression results correspond to Equation (7.6) in Chapter 7 of the book.

F tests of joint hypotheses in Python is very simple:

In [ ]:
hypotheses = 'str = 3, expn_stu = 0.5, el_pct=2'
test_result = result2.f_test(hypotheses)
print(test_result)

In [ ]:
print(result2.f_test('str = 3, expn_stu = 0.5, el_pct=2'))

In [ ]:
hypotheses = 'str = expn_stu, expn_stu*3=2*el_pct-4'
test_result = result2.f_test(hypotheses)
print(test_result)

#### **Quadratic function**

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)
dataset.head()

In [ ]:
dataset['avginc'].describe()

Now let's create a new variable/column that is a squared value of `avginc`:

In [ ]:
dataset['avginc2']=dataset['avginc']**2

In [ ]:
mod = smf.ols(formula='testscr ~ avginc + avginc2', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

In [ ]:
dataset['avginc3']=dataset['avginc']**3

In [ ]:
mod = smf.ols(formula='testscr ~ avginc + avginc2 + avginc3', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

In [ ]:
hypotheses = 'avginc2 = 0, avginc3 = 0'
test_result = result2.f_test(hypotheses)
print(test_result)

#### **Logarithm models**

Implementing linear-log, log-linear, or log-log models is easy, because the library `numpy` has a log function that you can directly deploy in the linear regressions (you can generate the log of a variable first, then run the regressions with this newly generated log variable; but here, I present it as a one-step process).

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)
dataset.head()

In [ ]:
dataset['ln_avginc']=numpy.log(dataset['avginc'])

In [ ]:
# The linear-log model
mod = smf.ols(formula='testscr ~ ln_avginc', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

In [ ]:
# The linear-log model
mod = smf.ols(formula='testscr ~ numpy.log(avginc)', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

In [ ]:
# The log-linear model
mod = smf.ols(formula='numpy.log(testscr) ~ avginc', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

In [ ]:
# The log-log model
mod = smf.ols(formula='numpy.log(testscr) ~ numpy.log(avginc)', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

#### **Interaction terms**

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)
dataset.head()

Now, let's generate binary variables `histr` and `hiel` as the textbook instructs:

In [ ]:
dataset['histr'] = (dataset['str'] >= 20).astype(int)
dataset['hiel'] = (dataset['el_pct'] >= 10).astype(int)

There is actually a very simple way (and, the preferable way) of running the regression with both `histr` and `hiel`, AND the interaction between `histr` and `hiel` (note that in the regression result below, `histr:hiel` is exactly the term (`histr` $\times$ `hiel`), admittedly a little confusing):

In [ ]:
mod = smf.ols(formula='testscr ~ histr*hiel', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

The above command is equivalent to:

In [ ]:
mod = smf.ols(formula='testscr ~ histr + hiel+ histr:hiel', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

The following analysis is about the effect of education on earnings. It replicates the results in **Table 8.1** in the textbook. The data set is [`ch8_cps.xlsx`](https://docs.google.com/spreadsheets/d/1c_OP5RVWNRLpjcu0sBfOMx4KoyWM6AZ8/edit?usp=drive_link&ouid=110351823593825164156&rtpof=true&sd=true) (or you can find the same link on Canvas course homepage).

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)
# if there is any error in this step (pandas.read_excel)
# try opening the Excel file on your local computer and saving it again.

dataset.head()

First we only keep those who are working and presumably not retired:

In [ ]:
dataset=dataset[(dataset['age'] >= 30) & (dataset['age'] <= 64)]

In [ ]:
print(dataset.describe())

**Column (1)** of **Table 8.1**:

In [ ]:
mod = smf.ols(formula='numpy.log(ahe) ~ yrseduc', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

**Column (2)** of **Table 8.1**:

In [ ]:
mod = smf.ols(formula='numpy.log(ahe) ~ female + yrseduc', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

**Column (3)** of **Table 8.1**:

In [ ]:
mod = smf.ols(formula='numpy.log(ahe) ~ female * yrseduc', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

**Column (4)** of **Table 8.1**:

In [ ]:
dataset['pexp']=dataset['age']-(dataset['yrseduc']+6)
dataset['pexp_2']=dataset['pexp']**2

In [ ]:
mod = smf.ols(formula='numpy.log(ahe) ~ female * yrseduc + pexp + pexp_2 + midwest+ south+west', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())

**Eq. (8.37) of test score regression on the interaction between student-teacher ratio and the percentage of English learners**

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
dataset = pandas.read_excel(file_path)
dataset.head()

In [ ]:
mod = smf.ols(formula='testscr ~ str * el_pct', data=dataset)
result = mod.fit()
result2 = result.get_robustcov_results(cov_type='HC1')
print(result2.summary())